# ATRIUM RSE Assessment Task

## Issues with the SSH Open Marketplace OpenAPI Specification

Whilst it is possible to interact the the marketplace API using simple HTTP requests it makes more sense
to use the [published OpenAPI specification](https://marketplace-api.sshopencloud.eu/v3/api-docs) to create
a custom client library which can validate both the responses from the API and any new data we wish to
submit to the marketplace.

The code in this notebook generates and uses a custom OpenAPI client library using the
[OpenAPI Generator from OpenAPITools](https://github.com/OpenAPITools/openapi-generator).

Unfortunately generating a client library which validates the API responses highlighted a problem with the
currently published documentation for the marketplace. Specifically the information relating to dates appears
to be invalid throughout the specification. For example, the `registrationDate` property of the `UserDto`
is defined as follows:

```
"registrationDate":{
    "pattern":"yyyy-MM-dd'T'HH:mm:ssZ",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

At first glance this looks sensible, and suggests that the date must conform to the `yyyy-MM-dd'T'HH:mm:ssZ`
format. The problem though, is that according to the OpenAPI specification the `pattern` field should be used
as a regular expression to validate the value of the string field. Is this case, that means that only the
pattern string itself would, I beleive, be a valid value for this field. According to the OpenAPI specification
the correct way to specify dates is by using the  `format` field. In this case that would result in the following:

```
"registrationDate":{
    "format": "date-time",
    "type":"string",
    "example":"2000-02-29T20:02:00+0000"
}
```

The OpenAPI specification states that valid values for this field must conform to `date-time` as defined by
[RFC3339](https://datatracker.ietf.org/doc/html/rfc3339#section-5.6), which includes the expected
`yyyy-MM-dd'T'HH:mm:ssZ` format.

I assume this was missed as the marketplace OpenAPI description, as currently published, is syntactically
valid just not logically correct.

In order to generate a useable client library I've produced [an updated version of the specification](api-docs.json)
which now correctly specifies dates.

## Initial Setup

We could have included the dependencies we need in a separate `requirements.txt` file but I've chosen instead to generate the file from within the notebook so that the dependencies we are using are made explicit and easy to view without having to switch to look at a separate file

In [ ]:
%%writefile requirements.txt

# this is used to generate the custom client library to aid in
# accessing the marketplace REST API
openapi-generator-cli==7.21.0

We can now install the dependencies, generate the custom client library, and then install that into the notebook environment

In [ ]:
# install all the dependencies we need for the rest of this notebook
%pip install -r requirements.txt

# generate a custom client library for working with the SSH Open Marketplace.
#
# note that we are using a local copy of the OpenAPI description for the marketplace
# due to the issue around dates. Once the official version has been updated this
# command couild be updated to use https://marketplace-api.sshopencloud.eu/v3/api-docs
! openapi-generator-cli generate -g python -i ./api-docs.json -o marketplace-client

# install the custom client library into the environment ready for us to use
%pip install ./marketplace-client

With the required dependencies installed we can now import and configure the custom client library

In [ ]:
import openapi_client

# By default the custom client library we created defaults to
# connecting to https://marketplace-api.sshopencloud.eu
# Should this need to be overriden a host parameter can be
# passed to the Configuration constructor. Note that this
# is where you would also specify authentication details etc.
configuration = openapi_client.Configuration()
